# L'ordre de mérite sur une année : d'où vient la série de prix

*Fabriquer soi-même, en cinq blocs de code exécutés, les 8 760 prix horaires de la France de 2030 : parc du TYNDP 2026 fixé, demande et météo d'une année, coût variable du gaz calculé à la main ; compter les heures où chaque filière fixe le prix, bouger le gaz et le CO₂ pour voir ce qui change et ce qui ne change pas, puis comparer à la série publiée du TYNDP.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/ordre-merite-annee.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets.

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), Path.cwd())  # la racine du dépôt des cas
if not (RACINE / "pommes_cases").is_dir():
    raise FileNotFoundError(f"pas de pommes_cases/ sous {RACINE} : placez ce notebook dans le dépôt "
                            "pommes-case-studies (à sa racine ou dans un de ses dossiers)")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")

Le premier modèle achetait au prix horaire de la France en 2030 publié par le TYNDP 2026, le plan décennal des réseaux européens, dans son scénario National Trends+ (NT+), qui suit les plans nationaux. Le texte sur la duale montrait que ce prix est la valeur d'un MWh de demande en plus. Ici, vous fabriquez la série vous-même : le parc de 2030 est fixé, le modèle choisit seulement quelle centrale tourne à chaque heure. C'est un cas d'école, pas une prévision.

Les valeurs datent du 23/11/2026, avec l'environnement du dépôt des cas : pommes 0.11.0, pommes_craft 0.6.0, linopy 0.10.1, HiGHS 1.15.1. Les blocs s'enchaînent dans une session : cinq résolutions de 1 à 2 s, 2 Go de mémoire.

## 1. Le parc et le coût variable de chaque centrale

Le coût variable d'une centrale est ce que coûte un MWh de plus : le combustible et le CO₂, divisés par le rendement (MWh d'électricité par MWh de combustible, au pouvoir calorifique inférieur, PCI), plus l'entretien proportionnel à la production. Pour le cycle combiné au gaz (CCGT), le gaz du TYNDP pour 2030 vaut 33,242 €/MWh PCI, le CO₂ 97,47 €/t, un MWh de gaz émet 0,2052 t de CO₂, et le rendement vaut 0,58 (cycles combinés français : 0,56 à 0,60) :

(33,242 + 0,2052 × 97,47) / 0,58 + 1,6 = 53,243 / 0,58 + 1,6 = 91,80 + 1,6 = **93,40 €/MWh**.

La turbine à combustion (TAC), moins efficace, coûte 53,243 / 0,42 + 1,6 = 128,37 €/MWh. Classées par coût variable croissant, les centrales forment l'ordre de mérite : chaque heure, on appelle les moins chères jusqu'à couvrir la demande, et le prix vaut le coût variable de la dernière appelée, le coût marginal du système. Un effacement est un consommateur payé pour réduire sa consommation ; le délestage coupe sans accord.

| Filière | Capacité (MW) | Coût variable (€/MWh) |
|---|---|---|
| Nucléaire | 63 020 | 15,63 |
| Éolien terrestre ; en mer | 30 500 ; 3 015 | 0 |
| Photovoltaïque (PV) au sol ; en toiture | 24 720 ; 22 269 | 0 |
| Hydraulique et autres | 28 102 | 0 |
| CCGT ; TAC | 6 553 ; 636 | 93,40 ; 128,37 |
| Fioul | 1 331 | 267,06 |
| Effacements | 5 675 | 350 |
| Délestage | sans limite | 3 000 |

Capacités, rendements, émissions et prix sont lus dans le TYNDP 2026 NT+ (version provisoire du 10/06/2026) ; les coûts variables en sont calculés ; le coût du délestage est une hypothèse des cas du dépôt. L'hydraulique (fil de l'eau, lacs) et les autres productions (biomasse, cogénération) reprennent la production horaire calculée par le TYNDP : le petit modèle ne pilote pas les lacs. Les stations de pompage et les électrolyseurs sont écartés.

In [ ]:
import sys
from pommes_cases.systeme_fr import parc

P = parc(2030)   # TYNDP 2026 NT+, France 2030 : capacités (MW), prix, rendements

def cout_variable(t, gaz=P["prix_gaz"], co2=P["prix_co2"]):
    combustible = {"ccgt": gaz, "ocgt": gaz, "fioul": P["prix_fioul"], "nucleaire": P["prix_nucleaire"]}[t]
    return (combustible + P[f"co2_{t}"] * co2) / P[f"rendement_{t}"] + P[f"variable_{t}"]

print({t: round(cout_variable(t), 2) for t in ("nucleaire", "ccgt", "ocgt", "fioul")})
# {'nucleaire': 15.63, 'ccgt': 93.4, 'ocgt': 128.37, 'fioul': 267.06}

## 2. Une année de demande et de météo

L'année météo WS003 est une année simulée par un modèle climatique ; le TYNDP lui donne le plus grand poids pour 2030 (0,63), et la série du premier modèle en vient. La demande est sa demande « native », sans voitures électriques, électrolyseurs ni pompes à chaleur hybrides (environ 39 TWh). Le vent et le soleil de la même année (base météo PECD 2026) plafonnent la production de chaque heure : c'est la disponibilité. Celle du nucléaire suit les arrêts pour maintenance : le dépôt l'estime d'après la production nucléaire du TYNDP, semaine par semaine.

In [ ]:
import logging
import numpy as np
import polars as pl
from pommes_cases.meteo import load_meteo, availability
from pommes_cases.systeme_fr import horaire, communs
logging.getLogger().setLevel(logging.ERROR)   # moins de messages

h = horaire(2030, "WS003")       # sorties horaires du TYNDP 2026 NT+
meteo = load_meteo("FR", 2030)   # facteurs de charge du PECD 2026, même année
def serie(valeurs, colonne):
    return pl.DataFrame({"hour": range(8760), "year_op": 2030, colonne: np.asarray(valeurs, float)})

demande = serie(h["demande_native"], "demand")
hydro = ["fil_eau", "lac", "autres_enr", "autres_non_enr"]
capacites = {t: P[f"capacite_{t}"] for t in ("nucleaire", "eolien_terrestre", "eolien_mer", "pv", "pv_toiture",
                                               "ccgt", "ocgt", "fioul", "effacement")}
capacites["hydro_autres"] = sum(P[f"capacite_{c}"] for c in hydro)
dispo = {t: availability(meteo, t) for t in ("eolien_terrestre", "eolien_mer", "pv", "pv_toiture")}
dispo["nucleaire"] = serie(communs(2030)["nucleaire_dispo"], "availability")
dispo["hydro_autres"] = serie(sum(h[c].to_numpy() for c in hydro) / capacites["hydro_autres"], "availability")
print(f"{h['demande_native'].sum() / 1e6:.1f} TWh, pointe {h['demande_native'].max() / 1e3:.1f} GW")
# 487.5 TWh, pointe 101.9 GW

## 3. Résoudre et lire 8 760 prix

Sans délestage explicite, pommes en ajouterait un gratuit et tous les prix tomberaient à zéro (premier modèle, bloc 5). Aucune énergie n'est jetée (écrêtement à capacité nulle) : une centrale en surplus produit sous sa disponibilité. Le prix d'une heure est la duale de son équilibre offre-demande (`operation_adequacy_constraint`), divisée par le facteur d'échelle de pommes (`dual_scale`), qui vaut 1 ici.

In [ ]:
from pommes_craft import (EnergyModel, Area, Demand, EconomicHypothesis, TimeStepManager,
                          LoadShedding, Spillage, ConversionTechnology)
from pommes.utils import dual_scale

def resoudre(gaz=P["prix_gaz"], co2=P["prix_co2"], defaillance=3000.0):
    couts = {t: cout_variable(t, gaz, co2) for t in ("nucleaire", "ccgt", "ocgt", "fioul")}
    couts["effacement"] = P["prix_effacement"]
    m = EnergyModel(name="ordre_merite", hours=list(range(8760)), year_ops=[2030], year_invs=[2030],
                    year_decs=[2055], modes=[], resources=["electricity"])
    with m.context():
        EconomicHypothesis("eco", discount_rate=0.0, year_ref=2030, planning_step=1)
        TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=8760.0)
        zone = Area("FR")
        zone.add_component(Demand(name="demande", resource="electricity", demand=demande))
        zone.add_component(LoadShedding(name="delestage", resource="electricity", cost=defaillance))
        zone.add_component(Spillage(name="ecretement", resource="electricity", max_capacity=0.0))
        for nom, mw in capacites.items():
            zone.add_component(ConversionTechnology(
                name=nom, factor={"electricity": 1.0}, variable_cost=couts.get(nom, 0.0),
                availability=dispo.get(nom, 1.0), life_span=25,
                power_capacity_investment_min=mw, power_capacity_investment_max=mw))
    m.run(store_linopy_model=True, store_pommes_parameters=True)
    lp = m.linopy_model
    prix = lp.dual["operation_adequacy_constraint"].squeeze().values / dual_scale(m.pommes_parameters).item()
    delestage = lp.solution["operation_load_shedding_power"].squeeze().values
    return prix + 0.0, couts, delestage   # + 0.0 efface le zéro négatif

prix, couts, delestage = resoudre()
print(np.sort(prix)[::-1][[0, 99, 499, 999, 8759]].round(2))   # 1re, 100e, 500e, 1 000e et dernière heure
# [3000.    350.     93.4    15.63    0.  ]
paliers = {"zero": 0.0, **couts, "delestage": 3000.0}
heures = {nom: int(np.isclose(prix, c).sum()) for nom, c in paliers.items()}
print(heures)
# {'zero': 88, 'nucleaire': 7698, 'ccgt': 597, 'ocgt': 28, 'fioul': 82, 'effacement': 198, 'delestage': 69}
print(f"moyenne {prix.mean():.2f} €/MWh, dont délestage {3000 * heures['delestage'] / 8760:.2f} ; "
      f"au plus {delestage.max() / 1e3:.1f} GW délestés")
# moyenne 54.55 €/MWh, dont délestage 23.63 ; au plus 12.3 GW délestés

Les 8 760 prix ne prennent que sept valeurs : la monotone (les heures classées du prix le plus haut au plus bas) est un escalier. La filière de la dernière centrale appelée est dite marginale. Le nucléaire l'est 7 698 heures : sans voisins à qui vendre, il baisse sa production pour suivre la demande. Aux 88 heures à zéro, vent, soleil et hydraulique suffisent. Le gaz fixe le prix 625 heures (597 + 28). En hiver, il manque jusqu'à 12,3 GW : les 69 heures délestées font 23,63 €/MWh, 43 % de la moyenne, à un prix choisi par le modélisateur.

## 4. Les curseurs : gaz et CO₂

Le gaz passe à 15 puis 65 €/MWh PCI, bornes du balayage du cas 0 ; le CO₂ à 50 puis 150 €/t, choisis autour des 97,47 du TYNDP.

In [ ]:
for gaz, co2 in [(15.0, P["prix_co2"]), (65.0, P["prix_co2"]), (P["prix_gaz"], 50.0), (P["prix_gaz"], 150.0)]:
    p, c, _ = resoudre(gaz, co2)
    print(f"{gaz:.2f} {co2:.2f}", [round(c[t], 2) for t in ("ccgt", "ocgt", "fioul")], round(p.mean(), 2),
          np.isclose(p, prix).sum())
# gaz, CO2, coûts [CCGT, TAC, fioul], prix moyen, heures inchangées
# 15.00 97.47 [61.95, 84.94, 267.06] 52.27 8135
# 65.00 97.47 [148.15, 203.98, 267.06] 58.52 8135
# 33.24 50.00 [76.6, 105.18, 228.98] 52.98 8053
# 33.24 150.00 [111.98, 154.03, 309.2] 56.29 8053

Ce qui bouge : le prix des 625 heures où le gaz est marginal et, avec le CO₂, des 82 heures du fioul. Ce qui ne bouge pas : les 8 135 autres heures (8 053 avec le CO₂), celles du nucléaire, des renouvelables, des effacements et du délestage. L'ordre de mérite ne change pas, donc ni la production ni les heures marginales. Du gaz à 15 au gaz à 65 €/MWh, la moyenne ne gagne que 6,25 €/MWh : la France isolée a rarement le gaz à la marge.

## 5. D'une petite série à une série publiée

Une série publiée sort d'un modèle de ce genre, nourri d'un parc, d'une demande, d'une météo, d'un gaz et d'un CO₂. Ces réglages, et le modèle lui-même, font que les quatre jeux de prix du cas 0 (séries horaires publiées) s'accordent sur la moyenne en 2030 et divergent en 2040. Comparez à celle du TYNDP, même année, même météo :

In [ ]:
from pommes_cases.prices import load_prices
tyndp = load_prices("tyndp2026", "NT+", "FR", 2030)["import_price"].to_numpy()
for s in (prix, tyndp):
    print(f"moyenne {s.mean():.2f}, médiane {np.median(s):.2f}, max {s.max():.0f}, au nucléaire "
          f"{np.isclose(s, couts['nucleaire'], atol=0.05).sum()}, valeurs {np.unique(s.round(2)).size}")
# moyenne 54.55, médiane 15.63, max 3000, au nucléaire 7698, valeurs 7
# moyenne 57.50, médiane 52.00, max 350, au nucléaire 1133, valeurs 1329

Les moyennes sont proches pour de mauvaises raisons : la vôtre doit 43 % de sa valeur au délestage, alors que la série du TYNDP ne dépasse pas 350 €/MWh, le prix des effacements. Sa médiane vaut 52 €/MWh, et elle n'a que 1 133 heures au coût du nucléaire, à 0,05 €/MWh près. Il manque au petit modèle :

- **les échanges** : reliée à ses voisins, la France partage leur prix, souvent celui d'une centrale au gaz, tant que les lignes ne sont pas saturées ; saturées à l'export, elles laissent son nucléaire fixer le prix ; elle importe à la pointe ;
- **le stockage, les lacs et les électrolyseurs pilotés**, qui consomment aux heures bon marché ou produisent aux heures chères ; la valeur de l'eau s'aligne sur le gaz qu'elle remplace.

Une centrale par filière donne sept paliers. Le TYNDP compte 1 329 valeurs distinctes au centime, dont 517 ne durent qu'une heure : centrales de rendements variés dans toute l'Europe, stocks, et petits écarts aux frontières (frais de passage, pertes). Reste l'écart au marché réel : les rigidités (puissance minimale des centrales, coût des démarrages), absentes de ce modèle, y font les heures bon marché [Une contrainte d'énergie n'est pas une rigidité : ce qu'un LP en prévision parfaite fait aux prix](https://www.robingirard.eu/assets/parcours-es/latest/textes/forme-des-prix.html).

## Vous l'avez compris quand…

- Vous retrouvez à la main 93,40 €/MWh (cycle combiné) et 128,37 €/MWh (turbine à combustion).
- Vous lisez au bloc 3 les heures où chaque filière fixe le prix (7 698 au nucléaire, 625 au gaz) et dites pourquoi le bloc 4 ne les change pas.
- Avant de lancer, vous prédisez la moyenne à gaz 65 €/MWh : 54,55 + (54,75 × 597 + 75,61 × 28) / 8 760 = 58,52 €/MWh.
- Vous prédisez 109,69 €/MWh pour un délestage à 10 000 €/MWh (54,55 + 69 × 7 000 / 8 760), puis le vérifiez.

## Pour aller plus loin

- [Le prix est une duale : trois centrales à la main, en LP, puis dans pommes_craft](https://www.robingirard.eu/assets/parcours-es/latest/textes/prix-dual.html)
- [Lire une série de prix horaires : six choses à regarder et les chiffres du cas 0](https://www.robingirard.eu/assets/parcours-es/latest/textes/lire-un-prix.html)
- [Dimensionner un parc par la méthode graphique : monotone de charge, droites de coût, heures de bascule](https://www.robingirard.eu/assets/parcours-es/latest/textes/txt-methode-graphique.html)
- [Cas 0 — Les scénarios de prix horaires : pourquoi divergent-ils ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/00_prix)

## Sources

1. [Dépôt pommes-case-studies (pommes_cases/systeme_fr.py, pommes_cases/meteo.py, data/systeme/README.md, 00_prix/hypotheses.yaml)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [ENTSO-E et ENTSOG, scénarios du TYNDP 2026 (National Trends+), version provisoire du 10/06/2026 : sorties horaires, PEMMDB, prix des combustibles et du CO₂, PECD](https://2026.entsos-tyndp-scenarios.eu/download/)
3. [Documentation pommes, description du modèle (contrainte d'adéquation)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
4. [README de pommes_craft sur PyPI (version 0.6.0)](https://pypi.org/project/pommes-craft/)